## Setup Libraries

In [1]:
!uv pip install -q pytabkit

from pytabkit import TabM_D_Classifier

In [2]:
import torch
import random
import warnings
import numpy as np, pandas as pd
from colorama import Fore, Style
from importlib.metadata import version
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import KBinsDiscretizer, TargetEncoder

warnings.filterwarnings('ignore')
print("PyTorch  version:", torch.__version__)

def seed_everything(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
seed_everything(42)

PyTorch  version: 2.11.0+cu128


## Load Data

In [10]:
## -- ⚠️ IMPORTANT: SELECT PLATFORM ⚠️ --

PLATFORM = "kaggle" # -> 'colab' 'kaggle'

if PLATFORM == "kaggle":
    PATH = "/kaggle/input/competitions/playground-series-s6e9/"
    submission = pd.read_csv(PATH+'sample_submission.csv')
    train = pd.read_csv(PATH+'train.csv')
    test = pd.read_csv(PATH+'test.csv')

    ORIG_PATH = "/kaggle/input/datasets/itzzomkar/ev-adoption-behavior-and-range-anxiety/"
    orig = pd.read_csv(ORIG_PATH+"EV_Adoption_and_Range_Anxiety_Dataset.csv")
elif PLATFORM == 'colab':
    from google.colab import drive
    drive.mount('/content/drive')

    PATH = "/content/drive/MyDrive/-- shared_notebooks --/Ps6e9 | EV Purchase/_ev_purchase_data/"
    submission = pd.read_csv(PATH+"sample_submission.csv")
    train = pd.read_csv(PATH+"train.csv")
    test = pd.read_csv(PATH+"test.csv")
    orig = pd.read_csv(PATH+"EV_Adoption_and_Range_Anxiety_Dataset.csv")

## =================================================================================

# ID     = "id"
# TARGET = 'Will_Buy_EV'

# obj = train.select_dtypes(include=["object", "category"]).columns.tolist()

# cat_cols  = [c for c in obj if c not in [TARGET, ID]]
# num_cols  = [c for c in train.columns if c not in cat_cols+[TARGET, ID]]
# base_cols = cat_cols + num_cols

# train[TARGET] = train[TARGET].map({'No': 0, 'Yes': 1})
# orig[TARGET]  = orig[TARGET].map({'No': 0, 'Yes': 1})

# FEATURES = [c for c in train.columns if c not in [ID, TARGET]]

# orig = orig[FEATURES+[TARGET]]
# print("Total Features:", len(FEATURES))

for (name, df) in {"Train": train, "Test": test , "Orig": orig}.items(): #
    print(f"{name} shape: {df.shape}")

print(f"\nTotal Nums: {len(num_cols)}")
print(f"Total Cats: {len(cat_cols)}")
print(f"Total base: {len(base_cols)}")

Train shape: (668665, 15)
Test shape: (286571, 14)
Orig shape: (10000, 15)

Total Nums: 11
Total Cats: 24
Total base: 13


In [12]:
# for c in num_cols:
#     orig[c] = orig[c].fillna(orig[c].median())

# print(orig.isna().sum())

## Preprocess Features

In [13]:
%%time
ID = 'id'
TARGET = 'Will_Buy_EV'
train[TARGET] = train[TARGET].map({'No': 0, 'Yes': 1})
orig[TARGET] = orig[TARGET].map({'No': 0, 'Yes': 1})
X = train.drop([ID, TARGET], axis=1); train_id = train[ID]
y = train[TARGET]
X_test = test.drop([ID], axis=1); test_id = test[ID]
del train, test
print("X      init shape:", X.shape)
print("X_test init shape:", X_test.shape, "\n")

cat_cols = X.select_dtypes(include=['object']).columns.tolist()
num_cols = X.select_dtypes(exclude=['object']).columns.tolist()
print("init len(cat_cols):", len(cat_cols))
print("init len(num_cols):", len(num_cols), "\n")

category_map = {}
important_combos = [
    ('Annual_Income_USD', 'Range_Anxiety_Level'),
    ('Age', 'Range_Anxiety_Level'),
    ('Annual_Income_USD', 'Income_/_100_floor_'),
]
def feature_engineering(df, fit=False):
    # Fill NaNs
    for col in cat_cols:
        df[col] = df[col].fillna("missing")
    for col in num_cols:
        df[col] = df[col].fillna(0.0)

    # Categorize string cats
    for col in cat_cols:
        if fit:
            codes, uniques = df[col].factorize()
            category_map[col] = uniques
        else:
            uniques = category_map[col]
            code_map = {cat: i for i, cat in enumerate(uniques)}
            codes = df[col].map(code_map).fillna(-1).astype('int32')
        df[col] = codes
        df[col] = df[col].astype('category')

    # Arithmetic interaction
    df['_Daily_Commute_km_/_Age'] = (df['Daily_Commute_km'] / (df['Age'] + 1e-6)).astype('float32')
    df['Income_/_100_floor_']  = np.floor(df['Annual_Income_USD'] / 100.0).astype('float32').astype('category')
    df['Income_/_1000_floor_']  = np.floor(df['Annual_Income_USD'] / 1000.0).astype('float32').astype('category')
    df['Income_/_10000_floor_']  = np.floor(df['Annual_Income_USD'] / 10000.0).astype('float32').astype('category')
    df['Daily_km_/_5_floor_']  = np.floor(df['Daily_Commute_km'] / 5.0).astype('float32').astype('category')

    # Categorize numericals
    for col in [i for i in num_cols if i not in ['Annual_Income_USD', 'Charging_Stations_Near_Home']]:
        cat_name = f"{col}_cat_"
        if fit:
            codes, uniques = np.floor(df[col]).factorize()
            category_map[col] = uniques
        else:
            uniques = category_map[col]
            code_map = {cat: i for i, cat in enumerate(uniques)}
            codes = np.floor(df[col]).map(code_map).fillna(-1).astype('int32')
        df[cat_name] = codes
        df[cat_name] = df[cat_name].astype('category')

    # Digit extraction
    for col in ['Daily_Commute_km']:
        decimal_name = f"_{col}_decimal"
        df[decimal_name] = (df[col] % 1).round(2).astype('float32')
    df['Annual_Income_USD_is_multiple_10_'] = (np.floor(df['Annual_Income_USD']) % 10 == 0).astype('category')

    # Target encoding from orig
    for col in ['Annual_Income_USD']:
        orig_enc_name = f"_{col}_mean_target_orig"
        df[orig_enc_name] = (
            df[col]
            .map(orig.groupby(col)[TARGET].mean())
            .fillna(orig[TARGET].mean())
            .astype('float32')
        )

    # Count encoding
    for col in ['Annual_Income_USD']:
        count_name = f"_{col}_count"
        if fit:
            count_map = df[col].value_counts()
            category_map[count_name] = count_map
        else:
            count_map = category_map[count_name]
        df[count_name] = df[col].astype(object).map(count_map).fillna(0).astype('int32')

    # Discretize numericals
    bin_config = {'Annual_Income_USD': [400, 600, 800, 900, 1100]}
    for col, bins_list in bin_config.items():
        for n_bins in bins_list:
            for strategy in ['quantile']:
                bin_name = f"{col}_{n_bins}_{strategy}_bin_"
                if fit:
                    kb = KBinsDiscretizer(
                        n_bins=n_bins,
                        encode='ordinal',
                        strategy=strategy,
                        subsample=None
                    )
                    binned = kb.fit_transform(df[[col]]).ravel().astype('int32')
                    category_map[bin_name] = kb
                else:
                    kb = category_map[bin_name]
                    binned = kb.transform(df[[col]]).ravel().astype('int32')
                df[bin_name] = binned
                df[bin_name] = df[bin_name].astype('category')

    # Create interaction categories
    combo_names = []
    for cols in important_combos:
        combo_name = '_'.join(cols) + '_'
        combo_names.append(combo_name)
        combo_series = df[cols[0]].astype(str)
        for col in cols[1:]:
            combo_series = combo_series + '_' + df[col].astype(str)
        if fit:
            codes, uniques = pd.factorize(combo_series, sort=False)
            category_map[combo_name] = uniques
        else:
            uniques = category_map[combo_name]
            code_map = {cat: i for i, cat in enumerate(uniques)}
            codes = combo_series.map(code_map).fillna(-1).astype('int32')
        df[combo_name] = codes
        df[combo_name] = df[combo_name].astype('category')   

    new_cat_cols = [col for col in df.columns if col.endswith('_')]
    new_num_cols = [col for col in df.columns if col.startswith('_')]
    return df, new_cat_cols, new_num_cols, combo_names

X, new_cat_cols, new_num_cols, combo_names = feature_engineering(X, fit=True)
X_test, _, _, _ = feature_engineering(X_test, fit=False)
cat_cols += new_cat_cols; num_cols += new_num_cols
print("len(new_cat_cols):", len(new_cat_cols))
print("len(new_num_cols):", len(new_num_cols), "\n")

print("prep len(cat_cols):", len(cat_cols))
print("prep len(num_cols):", len(num_cols), "\n")
print("X      prep shape:", X.shape)
print("X_test prep shape:", X_test.shape, "\n")

X

X      init shape: (668665, 13)
X_test init shape: (286571, 13) 

init len(cat_cols): 6
init len(num_cols): 7 

len(new_cat_cols): 18
len(new_num_cols): 4 

prep len(cat_cols): 24
prep len(num_cols): 11 

X      prep shape: (668665, 35)
X_test prep shape: (286571, 35) 

CPU times: user 4.41 s, sys: 273 ms, total: 4.69 s
Wall time: 4.69 s


,Age,Annual_Income_USD,Daily_Commute_km,Number_of_Cars_Owned,Charging_Stations_Near_Home,Charging_Stations_Near_Work,Environmental_Concern_Level,Gender,City_Type,Current_Car_Type,...,_Annual_Income_USD_mean_target_orig,_Annual_Income_USD_count,Annual_Income_USD_400_quantile_bin_,Annual_Income_USD_600_quantile_bin_,Annual_Income_USD_800_quantile_bin_,Annual_Income_USD_900_quantile_bin_,Annual_Income_USD_1100_quantile_bin_,Annual_Income_USD_Range_Anxiety_Level_,Age_Range_Anxiety_Level_,Annual_Income_USD_Income_/_100_floor__
0,66,92887.0,23.4,2,3,7,1.0,0,0,0,...,0.000000,96,214,321,427,482,584,0,0,0
1,38,30000.0,5.0,1,2,2,4.0,0,1,1,...,0.066427,61605,0,0,0,0,0,1,1,1
2,26,94389.0,36.8,1,8,15,5.0,1,2,0,...,1.000000,45,225,336,448,505,612,2,2,2
3,66,73580.0,23.7,2,6,9,3.0,0,0,2,...,1.000000,177,96,144,191,216,262,3,0,3
4,54,57898.0,50.8,1,2,3,3.0,0,0,2,...,0.500000,290,27,40,53,60,73,4,3,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
668660,30,71090.0,27.4,2,5,6,5.0,0,2,0,...,1.000000,81,80,119,159,180,218,9486,47,6094
668661,32,129990.0,5.0,2,5,4,1.0,1,0,1,...,0.000000,82,338,506,675,760,925,161,19,160
668662,64,121791.0,24.2,1,5,6,1.0,1,0,2,...,0.000000,175,321,480,640,721,876,3076,45,2836
668663,51,115923.0,43.7,2,0,0,1.0,1,1,1,...,0.175000,2,307,459,612,689,838,14312,16,9197


## Config

In [14]:
# realmlp_params = {
#     'random_state': 42,
#     'verbosity': 2,
#     'val_metric_name': '1-auc_ovr',

#     'n_ens': 8,
#     'n_epochs': 3,
#     'batch_size': 256,
#     'use_early_stopping': False,
#     'early_stopping_additive_patience': 10,
#     'early_stopping_multiplicative_patience': 1,

#     'lr': 0.06,
#     'wd': 0.0236,
#     'sq_mom': 0.988,
#     'lr_sched': 'lin_cos_log_15',
#     'first_layer_lr_factor': 0.25,

#     'embedding_size': 5,
#     'max_one_hot_cat_size': 18,
#     'hidden_sizes': [512, 256, 128],
#     'act': 'silu',
#     'p_drop': 0.05,
#     'p_drop_sched': 'expm4t',

#     'plr_hidden_1': 16,
#     'plr_hidden_2': 8,
#     'plr_act_name': 'gelu',
#     'plr_lr_factor': 0.1151,
#     'plr_sigma': 2.33,

#     'ls_eps': 0.01,
#     'ls_eps_sched': 'sqrt_cos',

#     'add_front_scale': False,
#     'bias_init_mode': 'neg-uniform-dynamic-2',
#     'tfms': ['one_hot', 'median_center', 'robust_scale',
#              'smooth_clip', 'embedding', 'l2_normalize'],
# }

params = {
    'random_state': 42,
    'verbosity': 2,
    'val_metric_name': '1-auc_ovr',
    
    'arch_type': 'tabm-mini',
    'num_emb_type': 'pwl',
    'allow_amp': True,
    # 'tabm_k': 24,

    'lr': 0.002,
    'n_epochs': 5,
    'patience': 3,

    'batch_size': 256,
    'd_block': 512,
    # 'n_blocks': 3,
    
    'd_embedding': 8,
    # 'dropout': 0.05,
    # 'num_emb_n_bins': 48,

    'compile_model': False,
    'gradient_clipping_norm': None,
    'tfms': ['quantile_tabr'],
    'weight_decay': 0.0,
}

params

{'random_state': 42,
 'verbosity': 2,
 'val_metric_name': '1-auc_ovr',
 'arch_type': 'tabm-mini',
 'num_emb_type': 'pwl',
 'allow_amp': True,
 'lr': 0.002,
 'n_epochs': 5,
 'patience': 3,
 'batch_size': 256,
 'd_block': 512,
 'd_embedding': 8,
 'compile_model': False,
 'gradient_clipping_norm': None,
 'tfms': ['quantile_tabr'],
 'weight_decay': 0.0}

## Train K-Fold

In [15]:
y.isna()

0         False
1         False
2         False
3         False
4         False
          ...  
668660    False
668661    False
668662    False
668663    False
668664    False
Name: Will_Buy_EV, Length: 668665, dtype: bool

In [16]:
%%time

USE_TE = True
FOLDS  = 3

skf = StratifiedKFold(n_splits=FOLDS, shuffle=True, random_state=42)
oof_preds = np.zeros(len(X))
test_preds = np.zeros(len(X_test))

for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y), 1):
    print("#"*35)
    print(f"### FOLD {fold}/{FOLDS}")
    print("#"*35)
    
    X_tr = X.iloc[tr_idx].copy()
    X_val = X.iloc[val_idx].copy()
    y_tr = y.iloc[tr_idx]
    y_val = y.iloc[val_idx]
    X_tst = X_test.copy()  

    # Target encoding
    if USE_TE:
        te_cols = combo_names
        print(f"⧗ Target encoding ({len(te_cols)}) features", end=" | ")
        TE = TargetEncoder(cv=5, smooth='auto', shuffle=True, random_state=0)
        tr_enc = TE.fit_transform(X_tr[te_cols], y_tr)
        val_enc = TE.transform(X_val[te_cols])
        tst_enc = TE.transform(X_tst[te_cols])

        te_names = [f"_{col}TE" for col in te_cols]
        X_tr[te_names] = tr_enc
        X_val[te_names] = val_enc
        X_tst[te_names] = tst_enc

    # if fold == 1: print(len(FEATURES), len(X_tr.columns.tolist()), "\n")
    print(f"train={X_tr.shape}, valid={X_val.shape}")

    model = TabM_D_Classifier(**params)
    model.fit(X_tr, y_tr, X_val, y_val,) # cat_col_names=cat_cols

    oof_preds[val_idx] = model.predict_proba(X_val)[:, 1]
    test_preds += model.predict_proba(X_tst)[:, 1] / FOLDS

    fold_score = roc_auc_score(y_val, oof_preds[val_idx])
    print(f"{Fore.YELLOW}{Style.BRIGHT}Fold {fold} | AUC: {fold_score:.6f}{Style.RESET_ALL}\n")

    torch.cuda.empty_cache()

oof_auc = np.round(roc_auc_score(y, oof_preds), 6)

print("\n" + "="*26)
print(f"FINAL OOF AUC: {Fore.BLUE}{Style.BRIGHT}{oof_auc}{Style.RESET_ALL}")
print("="*26)

###################################
### Fold 1/3 | shape=(445776, 38)
###################################
Columns classified as continuous: ['Age', 'Annual_Income_USD', 'Daily_Commute_km', 'Number_of_Cars_Owned', 'Charging_Stations_Near_Home', 'Charging_Stations_Near_Work', 'Environmental_Concern_Level', '_Daily_Commute_km_/_Age', '_Daily_Commute_km_decimal', '_Annual_Income_USD_mean_target_orig', '_Annual_Income_USD_count', '_Annual_Income_USD_Range_Anxiety_Level_TE', '_Age_Range_Anxiety_Level_TE', '_Annual_Income_USD_Income_/_100_floor__TE']
Columns classified as categorical: ['Gender', 'City_Type', 'Current_Car_Type', 'Home_Charging_Possible', 'Subsidy_Available', 'Range_Anxiety_Level', 'Income_/_100_floor_', 'Income_/_1000_floor_', 'Income_/_10000_floor_', 'Daily_km_/_5_floor_', 'Age_cat_', 'Daily_Commute_km_cat_', 'Number_of_Cars_Owned_cat_', 'Charging_Stations_Near_Work_cat_', 'Environmental_Concern_Level_cat_', 'Annual_Income_USD_is_multiple_10_', 'Annual_Income_USD_400_quantile

Epoch 0: 100%|██████████| 1742/1742 [17:26<00:00,  1.66it/s]


(val) -0.0585
🌸 New best epoch! 🌸



Epoch 1: 100%|██████████| 1742/1742 [17:33<00:00,  1.65it/s]


(val) -0.0641



...

(val) -0.0809



Epoch 4: 100%|██████████| 1742/1742 [17:31<00:00,  1.66it/s]


(val) -0.0819


Result:
{'val': -0.05846261978149414, 'epoch': 0}
Restoring best model
Fold 1 | AUC: 0.941535

###################################
### Fold 2/3 | shape=(445777, 38)
###################################
Columns classified as continuous: ['Age', 'Annual_Income_USD', 'Daily_Commute_km', 'Number_of_Cars_Owned', 'Charging_Stations_Near_Home', 'Charging_Stations_Near_Work', 'Environmental_Concern_Level', '_Daily_Commute_km_/_Age', '_Daily_Commute_km_decimal', '_Annual_Income_USD_mean_target_orig', '_Annual_Income_USD_count', '_Annual_Income_USD_Range_Anxiety_Level_TE', '_Age_Range_Anxiety_Level_TE', '_Annual_Income_USD_Income_/_100_floor__TE']
Columns classified as categorical: ['Gender', 'City_Type', 'Current_Car_Type', 'Home_Charging_Possible', 'Subsidy_Available', 'Range_Anxiety_Level', 'Income_/_100_floor_', 'Income_/_1000_floor_', 'Income_/_10000_floor_', 'Daily_km_/_5_floor_', 'Age_cat_', 'Daily_Commute_km_cat_', 'Number_of_Cars_Owned_cat_', 'Charging_Stations_Near_Work_

Epoch 0:  66%|██████▌   | 1150/1742 [11:34<05:57,  1.66it/s]


KeyboardInterrupt: 

## Evaluation and Submission

In [ ]:
oof_auc = roc_auc_score(y, oof_preds)

print("\n" + "="*26)
print(f"FINAL OOF AUC: {Fore.GREEN}{Style.BRIGHT}{oof_auc:.6f}{Style.RESET_ALL}")
print("="*26)

MODEL_NAME = "pytab_TABM"

# oof_df = pd.DataFrame({ID: train_id, TARGET: oof_preds})
# oof_df.to_csv('oof_preds.csv', index=False)

np.save(f"oof_{MODEL_NAME}_{oof_auc:.6f}.npy", oof_preds)
np.save(f"test_{MODEL_NAME}_{oof_auc:.6f}.npy", test_preds)

sub = pd.DataFrame({ID: test_id, TARGET: test_preds})
sub.to_csv('submission.csv', index=False)
sub.head()

In [ ]:
# !rm -r /kaggle/working